In [ ]:
# ---------- IMPORTS ----------
import os
import re
import csv
import numpy as np
import nibabel as nib
from typing import Optional, Dict, Any, List
import sys
import argparse  # NEW


# ---------- CONFIG ----------
NII_ROOT    = "/Volumes/M/HCP_EP/HCP_EP_concat_fMRI"
NPZ_ROOT    = "/Volumes/Genf/HCP_EP/HCP_EP_npz_files"

# Harvard–Oxford deterministic (thr50) MNI152 2mm paths (as you provided)
HO_CORT_PATH = "/Volumes/Genf/Atlases/Oxford-Cortical-Atlas/fsl/data/atlases/HarvardOxford/HarvardOxford-cortl-maxprob-thr50-2mm-without_background.nii.gz"
HO_SUB_PATH  = "/Volumes/Genf/Atlases/Oxford-Cortical-Atlas/fsl/data/atlases/HarvardOxford/HarvardOxford-sub-maxprob-thr50-2mm-only_subcortical_regions.nii.gz"

# Cerebellum atlas (unchanged)
CEREB_DIR = "/Volumes/Genf/Atlases/Cerebellum/atl-Anatom_space-MNI_dseg_resampled_to_MNI.nii"


def ensure_numpy(x):
    try:
        import torch
        if isinstance(x, torch.Tensor):
            return x.detach().cpu().numpy()
    except Exception:
        pass
    return x if isinstance(x, np.ndarray) else np.asarray(x)



# ---------- CORE ----------
def load_img(path, what):
    if not os.path.isfile(path):
        raise FileNotFoundError(f"{what} not found: {path}")
    try:
        return nib.load(path)
    except Exception as e:
        raise RuntimeError(f"Failed to load {what} at {path}: {e}")

def session_code_from_session_dir(session_dir_name: str) -> int:
    m = re.search(r'\b(AP|PA)[-_]?(\d{2})\b', session_dir_name, flags=re.IGNORECASE)
    if not m:
        raise ValueError(f"Cannot parse session id from pe name: '{session_dir_name}' (expected AP-01/PA-01/AP_02/...)")
    num = m.group(2)
    if num == "01":
        return 1
    if num == "02":
        return 2
    raise ValueError(f"Unsupported session number '{num}' in '{session_dir_name}' (only 01/02 supported).")

def _roi_timeseries_from_label_img(label_img: nib.Nifti1Image, X_2d: np.ndarray) -> np.ndarray:
    """Compute mean timecourse per ROI label in `label_img` over flattened fMRI X_2d=(V,T),
    excluding background label 0 and any labels that have no voxels.
    Returns an array shaped (T, K) where K is the number of kept labels.
    """
    # Labels
    L = np.asarray(label_img.dataobj, dtype=np.int32)
    V = L.size
    L_flat = L.reshape(V)

    # Count voxels per label
    max_label = int(L_flat.max())
    if max_label < 1:
        raise ValueError("Atlas contains no labels > 0.")
    counts = np.bincount(L_flat, minlength=max_label + 1).astype(np.float64)

    # Keep only labels >= 1 with at least one voxel
    keep_labels = np.where(counts[1:] > 0)[0] + 1  # shift back to original label indices
    if keep_labels.size == 0:
        raise ValueError("No non-zero labels with voxels found after excluding label 0.")

    Time = X_2d.shape[1]
    ts = np.empty((Time, keep_labels.size), dtype=np.float32)

    # For each timepoint, compute weighted sums and divide by counts only for kept labels
    for t in range(Time):
        sums_t = np.bincount(L_flat, weights=X_2d[:, t], minlength=max_label + 1).astype(np.float64)
        ts[t] = (sums_t[keep_labels] / counts[keep_labels]).astype(np.float32)

    return ts


def extract_for_harvard_oxford_and_cerebellum():

    # Load atlas images once
    ho_cort_img = load_img(HO_CORT_PATH, "Harvard-Oxford cortical (maxprob thr50, 2mm)")
    ho_sub_img  = load_img(HO_SUB_PATH,  "Harvard-Oxford subcortical (as provided)")
    cereb_img   = load_img(CEREB_DIR,    "Cerebellum atlas")

    # Sanity: same grid/affine across all atlases (and later fMRI)
    if ho_cort_img.shape != ho_sub_img.shape or ho_cort_img.shape != cereb_img.shape:
        raise ValueError(f"Atlas grid mismatch: cort {ho_cort_img.shape}, sub {ho_sub_img.shape}, cereb {cereb_img.shape}")
    if not (np.allclose(ho_cort_img.affine, ho_sub_img.affine, atol=1e-3) and
            np.allclose(ho_cort_img.affine, cereb_img.affine, atol=1e-3)):
        raise ValueError("Affine mismatch among atlas images.")

    atlas_shape = ho_cort_img.shape  # (X,Y,Z)
    V = np.prod(atlas_shape[:3])

    for subject in sorted(os.listdir(NII_ROOT)):
        subject_folder_in = os.path.join(NII_ROOT, subject)
        if not os.path.isdir(subject_folder_in):
            continue

        digits = ''.join(filter(str.isdigit, subject))
        if len(digits) < 4:
            raise ValueError(f"Session folder '{subject}' has fewer than 4 digits")
        sub_id_val = int(digits[-4:])

        for session in sorted(os.listdir(subject_folder_in)):
            sess_upper = session.upper()
            if not (sess_upper.startswith("AP_") or sess_upper.startswith("PA_") or
                    sess_upper.startswith("AP-") or sess_upper.startswith("PA-")):
                print(f"[SKIP] session '{session}' does not start with AP_/PA_ or AP-/PA-")
                continue
            session_folder_in = os.path.join(subject_folder_in, session)
            if not os.path.isdir(session_folder_in):
                print(f"[SKIP] no directory for session: {session_folder_in}")
                continue

            nii_path = os.path.join(session_folder_in, "concat_fMRI.nii.gz")
            if not os.path.isfile(nii_path):
                print(f"[SKIP] no concat_fMRI.nii.gz in {session_folder_in}")
                continue

            # --- Session-ID from folder name ---
            sess_code = session_code_from_session_dir(session)

            # --- Sub-session (AP/PA) from folder name ---
            sub_session_int = 1 if sess_upper.startswith("AP") else (2 if sess_upper.startswith("PA") else None)
            if sub_session_int is None:
                raise ValueError(f"Cannot parse sub_session from session folder name: '{session}'")

            npz_dir = os.path.join(NPZ_ROOT, subject, session)
            os.makedirs(npz_dir, exist_ok=True)
            npz_path = os.path.join(npz_dir, f"concat_fMRI_harvardoxford_timeseries.npz")
            if os.path.isfile(npz_path):
                print(f"[SKIP] already exists: {npz_path}")
                continue

            fmri_img = load_img(nii_path, "fMRI")
            if fmri_img.shape[:3] != atlas_shape[:3]:
                raise ValueError(f"Grid mismatch: fMRI {fmri_img.shape[:3]} vs atlases {atlas_shape[:3]}")
            if not np.allclose(fmri_img.affine, ho_cort_img.affine, atol=1e-3):
                raise ValueError("Affine mismatch between fMRI and atlases despite same shape.")

            # Flatten fMRI to (V,T)
            X = np.asarray(fmri_img.dataobj, dtype=np.float32)  # (X,Y,Z,T)
            Time = X.shape[3]
            X_2d = X.reshape(V, Time)

            # --- ROI time-series for each atlas ---
            ts_cort = _roi_timeseries_from_label_img(ho_cort_img, X_2d)   # (T, n_cort)
            ts_sub  = _roi_timeseries_from_label_img(ho_sub_img,  X_2d)   # (T, n_sub)
            ts_cere = _roi_timeseries_from_label_img(cereb_img,   X_2d)   # (T, n_cere)

            # Concatenate in this order: cortical | subcortical | cerebellum
            ts = np.concatenate([ts_cort, ts_sub, ts_cere], axis=1)

            # --- Per-time metadata (repeat subject-level values over time) ---
            time = np.arange(Time, dtype=np.float32).reshape(Time, 1)
            sub_id = np.full((Time,), sub_id_val, dtype=np.int32)
            session_id = np.full((Time,), sess_code, dtype=np.int32)
            sub_session = np.full((Time,), sub_session_int, dtype=np.int32)


            # Save combined atlas time series
            np.savez(
                npz_path,
                timeseries=ts,
                sub_id=sub_id,
                session_id=session_id,
                sub_session_id=sub_session,
            )

            print(f"[SAVED] T={Time}, N_rois={ts.shape[1]} (cort={ts_cort.shape[1]}, sub={ts_sub.shape[1]}, cere={ts_cere.shape[1]}), "
                  f"sub_id={sub_id_val}, session_id={sess_code}, sub_session_id={sub_session_int},  → {npz_path}")
            


def main():
    # Single pass: compute HO cortical + HO subcortical + cerebellum timecourses, concatenated
    extract_for_harvard_oxford_and_cerebellum()

if __name__ == "__main__":
    main()


--- STARTING VERIFICATION MODE (Read-Only) ---
[OK] MATCH Confirmed: ses-1001 / AP_01
[OK] MATCH Confirmed: ses-1001 / AP_02
[OK] MATCH Confirmed: ses-1001 / PA_01
[OK] MATCH Confirmed: ses-1001 / PA_02
[OK] MATCH Confirmed: ses-1002 / AP_01
[OK] MATCH Confirmed: ses-1002 / AP_02
[OK] MATCH Confirmed: ses-1002 / PA_01
[OK] MATCH Confirmed: ses-1002 / PA_02
[OK] MATCH Confirmed: ses-1003 / AP_01
[OK] MATCH Confirmed: ses-1003 / AP_02
[OK] MATCH Confirmed: ses-1003 / PA_01
[OK] MATCH Confirmed: ses-1003 / PA_02
[OK] MATCH Confirmed: ses-1004 / AP_01


KeyboardInterrupt: 

# For Schizophrenia

In [ ]:
# ---------- IMPORTS ----------
import os
import re
import csv
import numpy as np
import nibabel as nib
from typing import Optional, Dict, Any, List
import sys

# NEW: pandas for Excel
import pandas as pd

NII_ROOT    = "/Volumes/M/Geneva_Schizophrenia_Begue/concat_fmri"
NPZ_ROOT    = "/Volumes/Genf/Geneva_Schizophrenia_Begue/npz_files"
# Harvard–Oxford deterministic (thr50) MNI152 2mm paths (as you provided)
HO_CORT_PATH = "/Volumes/Genf/Atlases/Oxford-Cortical-Atlas/fsl/data/atlases/HarvardOxford/HarvardOxford-cortl-maxprob-thr50-2mm-without_background_resampled_79x95x79.nii.gz"
HO_SUB_PATH  = "/Volumes/Genf/Atlases/Oxford-Cortical-Atlas/fsl/data/atlases/HarvardOxford/HarvardOxford-sub-maxprob-thr50-2mm-only_subcortical_regions_resampled_79x95x79.nii.gz"

# Cerebellum atlas (unchanged)
CEREB_DIR = "/Volumes/Genf/Atlases/Cerebellum/Cerebellum_MNI_79x95x79-2mm.nii"


def ensure_numpy(x):
    try:
        import torch
        if isinstance(x, torch.Tensor):
            return x.detach().cpu().numpy()
    except Exception:
        pass
    return x if isinstance(x, np.ndarray) else np.asarray(x)

# ---------- CORE ----------
def load_img(path, what):
    if not os.path.isfile(path):
        raise FileNotFoundError(f"{what} not found: {path}")
    try:
        return nib.load(path)
    except Exception as e:
        raise RuntimeError(f"Failed to load {what} at {path}: {e}")

def session_code_from_session_dir(path_or_name: str) -> int:
    """
    Applies rules to the lowest folder name only (no AP/PA fallback):
      - If name starts with 'sub-0' -> session 1
      - If name starts with 'sub-G' -> session 1 if ends with T1, session 2 if ends with T2
      - Otherwise -> error
    Matching is case-insensitive.
    """
    # Use the lowest folder in the hierarchy
    name = os.path.basename(os.path.normpath(str(path_or_name))).strip()
    low = name.lower()

    if low.startswith("sub-0"):
        return 1

    if low.startswith("sub-g"):
        # allow ...T1 / ...T2 with or without separators
        if low.endswith("t2"):
            return 2
        if low.endswith("t1"):
            return 1
        raise ValueError(f"Cannot parse session id from '{name}': expected to end with T1 or T2.")

    raise ValueError(
        f"Cannot parse session id from '{name}'. "
        f"Expected lowest folder to start with 'sub-0' or 'sub-G' (optionally ending with T1/T2)."
    )


def _roi_timeseries_from_label_img(label_img: nib.Nifti1Image, X_2d: np.ndarray) -> np.ndarray:
    """Compute mean timecourse per ROI label in `label_img` over flattened fMRI X_2d=(V,T),
    excluding background label 0 and any labels that have no voxels.
    Returns an array shaped (T, K) where K is the number of kept labels.
    """
    # Labels
    L = np.asarray(label_img.dataobj, dtype=np.int32)
    V = L.size
    L_flat = L.reshape(V)

    # Count voxels per label
    max_label = int(L_flat.max())
    if max_label < 1:
        raise ValueError("Atlas contains no labels > 0.")
    counts = np.bincount(L_flat, minlength=max_label + 1).astype(np.float64)

    # Keep only labels >= 1 with at least one voxel
    keep_labels = np.where(counts[1:] > 0)[0] + 1  # shift back to original label indices
    if keep_labels.size == 0:
        raise ValueError("No non-zero labels with voxels found after excluding label 0.")

    Time = X_2d.shape[1]
    ts = np.empty((Time, keep_labels.size), dtype=np.float32)

    # For each timepoint, compute weighted sums and divide by counts only for kept labels
    for t in range(Time):
        sums_t = np.bincount(L_flat, weights=X_2d[:, t], minlength=max_label + 1).astype(np.float64)
        ts[t] = (sums_t[keep_labels] / counts[keep_labels]).astype(np.float32)

    return ts

def extract_for_harvard_oxford():

    # Load atlas images once
    ho_cort_img = load_img(HO_CORT_PATH, "Harvard-Oxford cortical (maxprob thr50, 2mm)")
    ho_sub_img  = load_img(HO_SUB_PATH,  "Harvard-Oxford subcortical (as provided)")
    cereb_img   = load_img(CEREB_DIR,    "Cerebellum atlas")

    # Sanity: same grid/affine across all atlases (and later fMRI)
    if ho_cort_img.shape != ho_sub_img.shape or ho_cort_img.shape != cereb_img.shape:
        raise ValueError(f"Atlas grid mismatch: cort {ho_cort_img.shape}, sub {ho_sub_img.shape}, cereb {cereb_img.shape}")
    if not (np.allclose(ho_cort_img.affine, ho_sub_img.affine, atol=1e-3) and
            np.allclose(ho_cort_img.affine, cereb_img.affine, atol=1e-3)):
        raise ValueError("Affine mismatch among atlas images.")

    atlas_shape = ho_cort_img.shape  # (X,Y,Z)
    V = np.prod(atlas_shape[:3])

    

    for subject in sorted(os.listdir(NII_ROOT)):
        subject_folder_in = os.path.join(NII_ROOT, subject)
        if not os.path.isdir(subject_folder_in):
            continue

        # Expect folder names like 'sub-XXXX'
        if not subject.lower().startswith("sub-"):
            raise ValueError(f"Subject folder '{subject}' does not start with 'sub-'")

        sub_id_val = subject.split("sub-", 1)[1]  # everything after 'sub-'

        if not sub_id_val:
            raise ValueError(f"Subject folder '{subject}' has no ID after 'sub-'")

        # Which session? (apply rule to the lowest folder)
        sess_code = session_code_from_session_dir(subject_folder_in)

        

        nii_path = os.path.join(subject_folder_in, "concat_fMRI.nii.gz")
        if not os.path.isfile(nii_path):
            print(f"[SKIP] no concat_fMRI.nii.gz in {subject_folder_in}")
            continue

        # --- Session-ID from folder name ---
        sess_code = session_code_from_session_dir(subject_folder_in)

        npz_dir = os.path.join(NPZ_ROOT, subject)
        os.makedirs(npz_dir, exist_ok=True)
        npz_path = os.path.join(npz_dir, f"concat_fMRI_harvardoxford_timeseries.npz")
        #if os.path.isfile(npz_path):
         #   print(f"[SKIP] {npz_path} file already exists in {subject_folder_in}")
          #  continue

        fmri_img = load_img(nii_path, "fMRI")
        if fmri_img.shape[:3] != atlas_shape[:3]:
            raise ValueError(f"Grid mismatch: fMRI {fmri_img.shape[:3]} vs atlases {atlas_shape[:3]}")
        if not np.allclose(fmri_img.affine, ho_cort_img.affine, atol=1e-3):
            raise ValueError("Affine mismatch between fMRI and atlases despite same shape.")

        # Flatten fMRI to (V,T)
        X = np.asarray(fmri_img.dataobj, dtype=np.float32)  # (X,Y,Z,T)
        Time = X.shape[3]
        X_2d = X.reshape(V, Time)

        # --- ROI time-series for each atlas ---
        ts_cort = _roi_timeseries_from_label_img(ho_cort_img, X_2d)   # (T, n_cort)
        ts_sub  = _roi_timeseries_from_label_img(ho_sub_img,  X_2d)   # (T, n_sub)
        ts_cere = _roi_timeseries_from_label_img(cereb_img,   X_2d)   # (T, n_cere)

        # Concatenate in this order: cortical | subcortical | cerebellum
        ts = np.concatenate([ts_cort, ts_sub, ts_cere], axis=1)

        # --- Per-time metadata (repeat subject-level values over time) ---
        time = np.arange(Time, dtype=np.float32).reshape(Time, 1)
        sub_id = np.full((Time,), sub_id_val, dtype=f"<U{max(16, len(sub_id_val))}")
        session_id = np.full((Time,), sess_code, dtype=np.int32)



        # Save combined atlas time series
        np.savez(
            npz_path,
            timeseries=ts,
            time=time,
            sub_id=sub_id,
            session_id=session_id,
        )

        print(
            f"[SAVED] T={Time}, N_rois={ts.shape[1]}, "
            f"sub_id={sub_id_val}, session_id={sess_code}, → {npz_path}"
        )

def main():
    extract_for_harvard_oxford()

if __name__ == "__main__":
    main()

--- STARTING VERIFICATION MODE (Read-Only) ---
Loading atlases...
[OK] MATCH Confirmed: sub-0101
[OK] MATCH Confirmed: sub-0105


KeyboardInterrupt: 

# Concatenate

In [ ]:
#!/usr/bin/env python3
import os
import numpy as np

# -------- CONFIG --------
NPZ_ROOT = "/Volumes/Genf/HCP_EP/HCP_EP_npz_files"

# Output prefixes (final files are saved at NPZ_ROOT/<PREFIX>.npz)
OUT_PREFIX_HO      = "all_subjects_concat_fMRI_harvardoxford_timeseries"
OUT_PREFIX_CEREB_HO= "all_subjects_concat_fMRI_cerebellum_timeseries"  # same filename as before, but includes CAINS if present

# Session order to scan (matches your generators)
SESS_ORDER = ["AP_01", "PA_01", "AP_02", "PA_02"]

# Filenames produced by the extractor script
F_HO      = "concat_fMRI_harvardoxford_timeseries.npz"
F_CEREB   = "concat_fMRI_cerebellum_timeseries.npz"


def load_npz(path):
    """Load one NPZ produced by the Harvard–Oxford + cerebellum extractor."""
    with np.load(path, allow_pickle=False) as z:
        ts   = z["timeseries"].astype(np.float32)    # (T, N_rois)
        # per-time metadata (all 1D unless noted)
        time = z["time"] if "time" in z else None     # (T,1) float, optional
        sub  = z["sub_id"].astype(np.int32)
        ses  = z["session_id"].astype(np.int32)
        sub_ses = z["sub_session_id"].astype(np.int32)
    return (ts, time, sub, ses, sub_ses)


def _concat_and_save(entries, out_path, label_for_log):
    """Concatenate collected lists and save the big NPZ."""
    if entries["files_found"] == 0:
        print(f"[SKIP {label_for_log}] No matching files found under {NPZ_ROOT}")
        return

    # Concatenate per-time arrays
    TS          = np.concatenate(entries["all_ts"], axis=0)
    SUB         = np.concatenate(entries["all_sub"], axis=0)
    SES         = np.concatenate(entries["all_ses"], axis=0)
    SUB_SES     = np.concatenate(entries["all_sub_ses"], axis=0)

    if entries["all_time"]:  # preserve 'time' if present in inputs
        TIME = np.concatenate(entries["all_time"], axis=0)
    else:
        TIME = None

    run_start = np.array(entries["run_start"], dtype=np.int64)
    run_len   = np.array(entries["run_len"],   dtype=np.int64)
    run_subid = np.array(entries["run_subid"], dtype=np.int32)
    run_subject_arr = np.array(entries["run_subject"], dtype="U64")
    run_session_arr = np.array(entries["run_session"], dtype="U16")

    print(f"Concatenated {label_for_log} TS shape:", TS.shape)

    # Build savez kwargs
    save_kwargs = dict(
        timeseries=TS,
        sub_id=SUB,
        session_id=SES,
        sub_session_id=SUB_SES,
        run_start=run_start,
        run_len=run_len,
        run_sub_id=run_subid,
        run_subject=run_subject_arr,
        run_session=run_session_arr,
    )
    if TIME is not None:
        save_kwargs["time"] = TIME

    np.savez(out_path, **save_kwargs)

    print(f"[SAVED {label_for_log}] {out_path}")
    print(f"[SUMMARY {label_for_log}] Total T={TS.shape[0]}, N_rois={TS.shape[1]}, runs={run_len.size}")


def _collect_init():
    return {
        "all_ts": [], "all_time": [],
        "all_sub": [], "all_ses": [], "all_sub_ses": [],
        "run_subject": [], "run_session": [], "run_subid": [], "run_start": [], "run_len": [],
        "files_found": 0,
    }


def process_harvard_oxford():
    """Concatenate all 'concat_fMRI_harvardoxford_timeseries.npz' files into one big NPZ (including CAINS)."""
    entries = _collect_init()
    out_path = os.path.join(NPZ_ROOT, f"{OUT_PREFIX_HO}.npz")

    for subject in sorted(os.listdir(NPZ_ROOT)):
        subject_dir = os.path.join(NPZ_ROOT, subject)
        if not os.path.isdir(subject_dir):
            continue

        for session in SESS_ORDER:
            session_dir = os.path.join(subject_dir, session)
            if not os.path.isdir(session_dir):
                continue

            npz_path = os.path.join(session_dir, F_HO)
            if not os.path.isfile(npz_path):
                continue

            (ts, time, sub, ses, sub_ses) = load_npz(npz_path)

            entries["files_found"] += 1

            start = sum(entries["run_len"])
            entries["all_ts"].append(ts)
            if time is not None:
                entries["all_time"].append(time)
            entries["all_sub"].append(sub)
            entries["all_ses"].append(ses)
            entries["all_sub_ses"].append(sub_ses)

            entries["run_subject"].append(subject)
            entries["run_session"].append(session)
            entries["run_subid"].append(int(sub[0]) if sub.size > 0 else -1)
            entries["run_start"].append(start)
            entries["run_len"].append(ts.shape[0])

            print(f"[ADD HO] {subject}/{session}: T={ts.shape[0]}, N_rois={ts.shape[1]}")

    _concat_and_save(entries, out_path, "HO")


def main():
    print("\n=== Concatenating Harvard–Oxford (cort+sub+cere) timeseries ===")
    process_harvard_oxford()



if __name__ == "__main__":
    main()


--- VERIFYING CONCATENATION: all_subjects_concat_fMRI_harvardoxford_timeseries ---
Re-aggregating source files from disk...
Concatenating in memory...
Loading existing file: /Volumes/Genf/HCP_EP/HCP_EP_npz_files/all_subjects_concat_fMRI_harvardoxford_timeseries.npz ...
[OK] MATCH Confirmed.
     Total Timepoints: 219384
     Runs processed: 554


'def main():\n    print("\n=== Concatenating Harvard–Oxford (cort+sub+cere) timeseries ===")\n    process_harvard_oxford()\n\n\n\nif __name__ == "__main__":\n    main()'

# For Schizophrenia

In [ ]:
#!/usr/bin/env python3
import os
import numpy as np

# -------- CONFIG --------
NPZ_ROOT = "/Volumes/Genf/Geneva_Schizophrenia_Begue/npz_files"

# Output prefix (final file saved as NPZ_ROOT/<PREFIX>.npz)
OUT_PREFIX_YB = "all_subjects_concat_fMRI_harvardoxford_timeseries"

# Filename produced by the extractor script (per subject)
F_YB = "concat_fMRI_harvardoxford_timeseries.npz"


def load_npz(path):
    """Load one NPZ produced by the current extractor."""
    with np.load(path, allow_pickle=False) as z:
        ts   = z["timeseries"].astype(np.float32)      # (T, N_rois)

        # Optional 'time'
        time = z["time"] if "time" in z else None      # (T,1) float or None

        # sub_id is STRING in your extractor
        sub  = z["sub_id"].astype(str)                 # (T,) <U...
        ses  = z["session_id"].astype(np.int32)        # (T,)                        # (T,) <U32

    return (ts, time, sub, ses)


def _concat_and_save(entries, out_path, label_for_log):
    """Concatenate collected lists and save the big NPZ."""
    if entries["files_found"] == 0:
        print(f"[SKIP {label_for_log}] No matching files found under {NPZ_ROOT}")
        return

    # Concatenate per-time arrays
    TS           = np.concatenate(entries["all_ts"], axis=0)
    SUB          = np.concatenate(entries["all_sub"], axis=0).astype("U64")
    SES          = np.concatenate(entries["all_ses"], axis=0)

    if entries["all_time"]:  # preserve 'time' if present in inputs
        TIME = np.concatenate(entries["all_time"], axis=0)
    else:
        TIME = None

    run_start   = np.array(entries["run_start"], dtype=np.int64)
    run_len     = np.array(entries["run_len"],   dtype=np.int64)
    run_subid   = np.array(entries["run_subid"], dtype="U64")
    run_subject = np.array(entries["run_subject"], dtype="U64")
    run_session = np.array(entries["run_session"], dtype="U16")

    print(f"Concatenated {label_for_log} TS shape:", TS.shape)

    # Build savez kwargs
    save_kwargs = dict(
        timeseries=TS,
        sub_id=SUB,
        session_id=SES,
        run_start=run_start,
        run_len=run_len,
        run_sub_id=run_subid,
        run_subject=run_subject,
        run_session=run_session,
    )
    if TIME is not None:
        save_kwargs["time"] = TIME

    np.savez(out_path, **save_kwargs)

    print(f"[SAVED {label_for_log}] {out_path}")
    print(f"[SUMMARY {label_for_log}] Total T={TS.shape[0]}, N_rois={TS.shape[1]}, runs={run_len.size}")


def _collect_init():
    return {
        "all_ts": [], "all_time": [],
        "all_sub": [],
        "run_subject": [], "run_session": [], "run_subid": [], "run_start": [], "run_len": [],
        "files_found": 0,
    }


def process_harvard_oxford():
    """
    Concatenate all '<subject>/<F_YB>' files into one big NPZ.
    (No session subfolders — each subject has a single NPZ.)
    """
    entries = _collect_init()
    out_path = os.path.join(NPZ_ROOT, f"{OUT_PREFIX_YB}.npz")

    for subject in sorted(os.listdir(NPZ_ROOT)):
        subject_dir = os.path.join(NPZ_ROOT, subject)
        if not os.path.isdir(subject_dir):
            continue

        npz_path = os.path.join(subject_dir, F_YB)
        if not os.path.isfile(npz_path):
            continue

        (ts, time, sub, ses) = load_npz(npz_path)

        entries["files_found"] += 1

        start = sum(entries["run_len"])
        entries["all_ts"].append(ts)
        if time is not None:
            entries["all_time"].append(time)
        entries["all_sub"].append(sub)
        entries["all_ses"].append(ses)

        # Run-level bookkeeping
        entries["run_subject"].append(subject)
        # Your extractor saves a session_id vector (all equal per subject). Store as small string.
        entries["run_session"].append(str(int(ses[0]) if ses.size > 0 else -1))
        entries["run_subid"].append(str(sub[0]) if sub.size > 0 else "")
        entries["run_start"].append(start)
        entries["run_len"].append(ts.shape[0])

        print(f"[ADD YB] {subject}: T={ts.shape[0]}, N_rois={ts.shape[1]}")

    _concat_and_save(entries, out_path, "YB")


def main():
    print("\n=== Concatenating Harvard–Oxford timeseries (one NPZ per subject) ===")
    process_harvard_oxford()


if __name__ == "__main__":
    main()

--- VERIFYING CONCATENATION: all_subjects_concat_fMRI_harvardoxford_timeseries ---
Re-aggregating source files from disk...
Concatenating...
Loading existing file: /Volumes/Genf/Geneva_Schizophrenia_Begue/npz_files/all_subjects_concat_fMRI_harvardoxford_timeseries.npz ...
[OK] MATCH Confirmed.
     Total Timepoints: 92507
     Subjects processed: 160


'def main():\n    print("\n=== Concatenating Harvard–Oxford timeseries (one NPZ per subject) ===")\n    process_harvard_oxford()\n\n\nif __name__ == "__main__":\n    main()'